# Important:

In the CSV file, set the column including the names to "Name", and the column including the phone numbers to "Phone Number", otherwise this script will not work.

Put one .csv and one .txt file in the same folder as this notebook. The text file is the message. Write [name] where the person's name should be filled in.

This script uses Google Chrome, Python, and the selenium and pandas packages.

Login to WhatsApp in the Chrome window this script opens before running the next cell. Scan the QR code on the WhatsApp Web page.

## To install the necessary packages:
```python
pip install pandas selenium
```

## Manual:

Add one CSV of contacts and one text file of the message into the same folder as this notebook.

The CSV looks like this:

```
Name,Phone Number
Ada,85291234567
```

The text file looks like this:

```
Hello [name], this is a reminder.
```

When the script starts, input an image path if you want to send an image, for example "C:/User/Image.jpeg". If you do not want an image, input NA.

Then wait for the program to finish. The time needed depends on your internet speed and the length of your list.



In [ ]:
from selenium import webdriver 
from selenium.webdriver.common.by import By 
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.support.ui import WebDriverWait 
import time 
import pandas as pd 
from pathlib import Path
from selenium.webdriver.support import expected_conditions as EC 

folders = []
nested = Path("Whatsapp Message Automater")
if nested.is_dir():
    folders.append(nested)
folders.append(Path.cwd())

folder = None
csv_files = []
text_files = []
seen = set()
for candidate in folders:
    resolved = candidate.resolve()
    if resolved in seen:
        continue
    seen.add(resolved)
    found_csv = sorted(resolved.glob("*.csv"))
    found_text = sorted(resolved.glob("*.txt"))
    if found_csv or found_text:
        folder = resolved
        csv_files = found_csv
        text_files = found_text
        break

if folder is None or len(csv_files) != 1 or len(text_files) != 1:
    raise FileNotFoundError("Put one .csv and one .txt in the same folder as this notebook.")

phonenumberList = pd.read_csv(csv_files[0], dtype=str)
message_template = text_files[0].read_text(encoding="utf-8").strip()
print(csv_files[0].name)
print(text_files[0].name)

imagefileinput = input("image file here (including pathway, if no image, input NA):") 

options = webdriver.ChromeOptions()
options.page_load_strategy = "eager"
driver = webdriver.Chrome(options=options)
driver.get("https://web.whatsapp.com")
print("please log into your whatsapp account, then run the next program")

## For Image Output, Sending to a group of people only

In [ ]:
def open_chat(number):
    """Open a chat, clicking through the Open app / Continue to WhatsApp Web page when it appears."""
    phone = str(number).strip().lstrip("+").replace(" ", "").replace("-", "")
    driver.get(
        f"https://web.whatsapp.com/send/?phone={phone}&text&type=phone_number&app_absent=1"
    )
    continue_xpath = (
        "//*[(self::a or self::button or @role='button') and ("
        "contains(normalize-space(.), 'Continue to WhatsApp Web')"
        " or contains(normalize-space(.), 'Continue to chat'))]"
    )

    def chat_or_continue(current):
        if current.find_elements(By.XPATH, '//div[@data-tab="10"]'):
            return "chat"
        buttons = [
            button
            for button in current.find_elements(By.XPATH, continue_xpath)
            if button.is_displayed()
        ]
        if buttons:
            return min(buttons, key=lambda button: len(button.text or ""))
        return False

    def continue_in_this_tab(button):
        """Follow Continue to WhatsApp Web here. A normal click opens a new tab."""
        original = driver.current_window_handle
        before = set(driver.window_handles)
        href = driver.execute_script(
            """
            const el = arguments[0];
            const link = el.closest('a') || el.querySelector('a');
            return (link && link.href) || el.href || null;
            """,
            button,
        )
        if href:
            driver.get(href)
            return
        driver.execute_script(
            """
            window.open = function(url) {
                if (url) window.location.assign(url);
                return window;
            };
            const el = arguments[0];
            const link = el.closest('a');
            if (link) link.removeAttribute('target');
            el.removeAttribute('target');
            el.click();
            """,
            button,
        )
        opened = [handle for handle in driver.window_handles if handle not in before]
        if opened:
            driver.switch_to.window(opened[-1])
            url = driver.current_url
            driver.close()
            driver.switch_to.window(original)
            driver.get(url)

    for _ in range(2):
        found = WebDriverWait(driver, 120).until(chat_or_continue)
        if found == "chat":
            return
        continue_in_this_tab(found)
    WebDriverWait(driver, 120).until(
        EC.presence_of_element_located((By.XPATH, '//div[@data-tab="10"]'))
    )


def type_one_message(box, message):
    """Put the whole template in one box. A plain newline would press Enter and send each line."""
    driver.execute_script(
        """
        const box = arguments[0];
        const text = arguments[1];
        box.focus();
        try {
            const data = new DataTransfer();
            data.setData('text/plain', text);
            box.dispatchEvent(new ClipboardEvent('paste', {
                bubbles: true,
                cancelable: true,
                clipboardData: data,
            }));
        } catch (err) {}
        """,
        box,
        message,
    )
    inserted = driver.execute_script("return (arguments[0].innerText || '').trim()", box)
    if not inserted:
        driver.execute_script(
            """
            const box = arguments[0];
            box.focus();
            const selection = window.getSelection();
            const range = document.createRange();
            range.selectNodeContents(box);
            range.collapse(false);
            selection.removeAllRanges();
            selection.addRange(range);
            document.execCommand('insertText', false, arguments[1]);
            """,
            box,
            message,
        )
        inserted = driver.execute_script("return (arguments[0].innerText || '').trim()", box)
    if inserted:
        return
    lines = message.splitlines()
    for index, line in enumerate(lines):
        if line:
            box.send_keys(line)
        if index < len(lines) - 1:
            box.send_keys(Keys.chord(Keys.SHIFT, Keys.ENTER))


import base64
import ctypes
import subprocess
import time
from ctypes import wintypes
from selenium.webdriver.common.action_chains import ActionChains


def copy_image_to_clipboard(path):
    encoded = base64.b64encode(str(path).encode("utf-8")).decode("ascii")
    subprocess.run(
        [
            "powershell", "-NoProfile", "-STA", "-Command",
            "Add-Type -AssemblyName System.Windows.Forms; "
            "Add-Type -AssemblyName System.Drawing; "
            f"$path = [Text.Encoding]::UTF8.GetString([Convert]::FromBase64String('{encoded}')); "
            "$img = [System.Drawing.Image]::FromFile($path); "
            "[System.Windows.Forms.Clipboard]::SetImage($img); "
            "$img.Dispose()",
        ],
        check=True,
    )


def copy_text_to_clipboard(text):
    if text is None or str(text) == "":
        raise ValueError("The message is empty, so it cannot be copied.")
    user32 = ctypes.windll.user32
    kernel32 = ctypes.windll.kernel32
    user32.OpenClipboard.argtypes = [wintypes.HWND]
    user32.OpenClipboard.restype = wintypes.BOOL
    user32.EmptyClipboard.restype = wintypes.BOOL
    user32.SetClipboardData.argtypes = [wintypes.UINT, wintypes.HANDLE]
    user32.SetClipboardData.restype = wintypes.HANDLE
    user32.CloseClipboard.restype = wintypes.BOOL
    kernel32.GlobalAlloc.argtypes = [wintypes.UINT, ctypes.c_size_t]
    kernel32.GlobalAlloc.restype = wintypes.HGLOBAL
    kernel32.GlobalLock.argtypes = [wintypes.HGLOBAL]
    kernel32.GlobalLock.restype = ctypes.c_void_p
    kernel32.GlobalUnlock.argtypes = [wintypes.HGLOBAL]
    kernel32.GlobalUnlock.restype = wintypes.BOOL
    kernel32.GlobalFree.argtypes = [wintypes.HGLOBAL]
    kernel32.GlobalFree.restype = wintypes.HGLOBAL
    data = str(text).encode("utf-16-le") + b"\x00\x00"
    opened = False
    for _ in range(10):
        if user32.OpenClipboard(None):
            opened = True
            break
        time.sleep(0.1)
    if not opened:
        raise RuntimeError("Could not open the clipboard.")
    handle = None
    try:
        if not user32.EmptyClipboard():
            raise RuntimeError("Could not clear the clipboard.")
        handle = kernel32.GlobalAlloc(0x0002, len(data))
        if not handle:
            raise RuntimeError("Could not allocate clipboard memory.")
        locked = kernel32.GlobalLock(handle)
        if not locked:
            raise RuntimeError("Could not lock clipboard memory.")
        ctypes.memmove(locked, data, len(data))
        kernel32.GlobalUnlock(handle)
        if not user32.SetClipboardData(13, handle):
            raise RuntimeError("Could not put the message on the clipboard.")
        handle = None
    finally:
        if handle:
            kernel32.GlobalFree(handle)
        user32.CloseClipboard()


def paste_into(box):
    driver.execute_script(
        """
        const box = arguments[0];
        box.scrollIntoView({block: 'center'});
        box.focus();
        box.click();
        """,
        box,
    )
    ActionChains(driver).key_down(Keys.CONTROL).send_keys("v").key_up(Keys.CONTROL).perform()


if imagefileinput != "NA":

    image_path = str(Path(imagefileinput.strip().strip('"').strip("'")).expanduser().resolve())
    if not Path(image_path).is_file():
        raise FileNotFoundError(f"Image file not found: {image_path}")

    for name, number in zip(phonenumberList["Name"], phonenumberList["Phone Number"]):
        final_message = message_template.replace("[name]", str(name).strip())
        open_chat(number)
        copy_image_to_clipboard(image_path)
        chat_box = driver.find_element(By.XPATH, '//div[@data-tab="10"]')
        paste_into(chat_box)
        time.sleep(3)
        if final_message:
            copy_text_to_clipboard(final_message)
            caption = WebDriverWait(driver, 15).until(EC.presence_of_element_located((
                By.CSS_SELECTOR, '[data-testid="media-caption-input-container"]'
            )))
            paste_into(caption)
            time.sleep(1)
        send = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((
            By.XPATH,
            '//div[@role="button" and starts-with(@aria-label, "Send") and contains(@aria-label, "selected")]'
        )))
        driver.execute_script("arguments[0].click();", send)
        time.sleep(5)

else:

    for name, number in zip(phonenumberList["Name"], phonenumberList["Phone Number"]):
        open_chat(number)
        time.sleep(1)
        messageInput = driver.find_element(By.XPATH, '//div[@data-tab="10"]')
        final_message = message_template.replace("[name]", str(name).strip())
        type_one_message(messageInput, final_message)
        time.sleep(1)
        messageInput.send_keys(Keys.ENTER)
        time.sleep(1)


## Text-only demo, one chat

Run this cell by itself. It opens Chrome and waits while you log in to WhatsApp Web. It does not read the CSV, the text file, or the image cell.

Use your own number. It sends plain text at most 5 times, then stops.


In [ ]:
def open_chat(number):
    """Open a chat, clicking through the Open app / Continue to WhatsApp Web page when it appears."""
    phone = str(number).strip().lstrip("+").replace(" ", "").replace("-", "")
    driver.get(
        f"https://web.whatsapp.com/send/?phone={phone}&text&type=phone_number&app_absent=1"
    )
    continue_xpath = (
        "//*[(self::a or self::button or @role='button') and ("
        "contains(normalize-space(.), 'Continue to WhatsApp Web')"
        " or contains(normalize-space(.), 'Continue to chat'))]"
    )

    def chat_or_continue(current):
        if current.find_elements(By.XPATH, '//div[@data-tab="10"]'):
            return "chat"
        buttons = [
            button
            for button in current.find_elements(By.XPATH, continue_xpath)
            if button.is_displayed()
        ]
        if buttons:
            return min(buttons, key=lambda button: len(button.text or ""))
        return False

    def continue_in_this_tab(button):
        """Follow Continue to WhatsApp Web here. A normal click opens a new tab."""
        original = driver.current_window_handle
        before = set(driver.window_handles)
        href = driver.execute_script(
            """
            const el = arguments[0];
            const link = el.closest('a') || el.querySelector('a');
            return (link && link.href) || el.href || null;
            """,
            button,
        )
        if href:
            driver.get(href)
            return
        driver.execute_script(
            """
            window.open = function(url) {
                if (url) window.location.assign(url);
                return window;
            };
            const el = arguments[0];
            const link = el.closest('a');
            if (link) link.removeAttribute('target');
            el.removeAttribute('target');
            el.click();
            """,
            button,
        )
        opened = [handle for handle in driver.window_handles if handle not in before]
        if opened:
            driver.switch_to.window(opened[-1])
            url = driver.current_url
            driver.close()
            driver.switch_to.window(original)
            driver.get(url)

    for _ in range(2):
        found = WebDriverWait(driver, 120).until(chat_or_continue)
        if found == "chat":
            return
        continue_in_this_tab(found)
    WebDriverWait(driver, 120).until(
        EC.presence_of_element_located((By.XPATH, '//div[@data-tab="10"]'))
    )


def type_one_message(box, message):
    """Put the whole template in one box. A plain newline would press Enter and send each line."""
    driver.execute_script(
        """
        const box = arguments[0];
        const text = arguments[1];
        box.focus();
        try {
            const data = new DataTransfer();
            data.setData('text/plain', text);
            box.dispatchEvent(new ClipboardEvent('paste', {
                bubbles: true,
                cancelable: true,
                clipboardData: data,
            }));
        } catch (err) {}
        """,
        box,
        message,
    )
    inserted = driver.execute_script("return (arguments[0].innerText || '').trim()", box)
    if not inserted:
        driver.execute_script(
            """
            const box = arguments[0];
            box.focus();
            const selection = window.getSelection();
            const range = document.createRange();
            range.selectNodeContents(box);
            range.collapse(false);
            selection.removeAllRanges();
            selection.addRange(range);
            document.execCommand('insertText', false, arguments[1]);
            """,
            box,
            message,
        )
        inserted = driver.execute_script("return (arguments[0].innerText || '').trim()", box)
    if inserted:
        return
    lines = message.splitlines()
    for index, line in enumerate(lines):
        if line:
            box.send_keys(line)
        if index < len(lines) - 1:
            box.send_keys(Keys.chord(Keys.SHIFT, Keys.ENTER))

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

if "driver" in globals():
    try:
        driver.quit()
    except Exception:
        pass

options = webdriver.ChromeOptions()
options.page_load_strategy = "eager"
driver = webdriver.Chrome(options=options)
driver.get("https://web.whatsapp.com")
input("Log in to WhatsApp Web in the Chrome window, then press Enter: ")

DEMO_LIMIT = 100
phone = input("Your WhatsApp number, with country code: ").strip()
message = input("Text message: ").strip()
raw_count = input(f"How many times (1-{DEMO_LIMIT}): ").strip()

digits = phone.lstrip("+").replace(" ", "").replace("-", "")
if not digits.isdigit() or len(digits) < 8:
    raise ValueError("Enter your number with the country code, digits only.")
if message == "":
    raise ValueError("Enter a text message.")
if not raw_count.isdigit() or not 1 <= int(raw_count) <= DEMO_LIMIT:
    raise ValueError(f"Enter a whole number from 1 to {DEMO_LIMIT}.")
count = int(raw_count)

open_chat(digits)
for sent in range(1, count + 1):
    message_box = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.XPATH, '//div[@data-tab="10"]'))
    )
    type_one_message(message_box, message)
    time.sleep(0.5)
    message_box.send_keys(Keys.ENTER)
    print(f"Sent {sent} of {count}")
    time.sleep(0.5)
